In [1]:
import pandas as pd, numpy as np, mlflow, mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt, seaborn as sns


# ── 1. Load split data ──────────────────────────────────────────────
X_train = pd.read_csv(r'C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data\train\X_train.csv')
X_test  = pd.read_csv(r'C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data\test\X_test.csv')
y_train = pd.read_csv(r'C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data\train\y_train.csv').squeeze()
y_test  = pd.read_csv(r'C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data\test\y_test.csv').squeeze()

c:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\bank_mkt_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ── 2. Configure MLflow ─────────────────────────────────────────────
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('bank-marketing-classifier')

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1780356359584, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780356359584, lifecycle_stage='active', name='bank-marketing-classifier', tags={}, trace_location=None, workspace='default'>

In [6]:
# ── 3. Train inside a run ───────────────────────────────────────────
with mlflow.start_run(run_name='gbc-baseline') as run:
    # Hyperparameters
    params = {
        'n_estimators': 200,
        'learning_rate': 0.1,
        'max_depth': 4,
        'subsample': 0.8,
        'random_state': 42
    }
    mlflow.log_params(params)
    # Train
    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)
    # Evaluate
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
    }
    mlflow.log_metrics(metrics)
    # Log model artifact
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path='model',
        registered_model_name='BankMarketingClassifier',
        input_example=X_test.head(5)
    )
    
    run_id = run.info.run_id
    print(f'Run ID: {run_id}')
    print(f'ROC-AUC: {metrics["roc_auc"]:.4f}')
    print(f'F1:      {metrics["f1"]:.4f}')


2026/06/03 14:46:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 14:46:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\bank_mkt_venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (t

Run ID: ddf0beb3861c459f9351bb4874e6a16f
ROC-AUC: 0.8136
F1:      0.3832
🏃 View run gbc-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/ddf0beb3861c459f9351bb4874e6a16f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'BankMarketingClassifier'.
